# LeetCode 1086 — High Five bằng Python

**Link đề:** https://leetcode.com/problems/high-five/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Sorting, Heap (Priority Queue)

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def highFive(self, items: list[list[int]]) -> list[list[int]]:
        ...
```

## Mục tiêu học
1. Hiểu yêu cầu: **gom điểm theo học sinh**, lấy **5 điểm cao nhất**, tính **trung bình chia lấy nguyên**, xuất theo ID tăng dần.
2. Dùng `dict` / `collections.defaultdict(list)` để **gom nhóm** (group by).
3. Sắp giảm dần `sorted(..., reverse=True)`, cắt `[:5]`, chia nguyên `//`.
4. Học **heap** (`heapq`) — giữ "top k" với min-heap cỡ 5, và cách **sắp toàn bộ một lần** theo `(id, -score)`.
5. Trace, chứng minh, so sánh Time/Space, các lỗi hay gặp, bài tập mở rộng.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho danh sách `items`, mỗi phần tử `items[i] = [ID_i, score_i]` là **một lần điểm** của học sinh có mã `ID_i`.

Với **mỗi học sinh**, tính **trung bình 5 điểm cao nhất** (top five average) = tổng 5 điểm cao nhất `// 5` (**chia lấy phần nguyên**).

Trả về mảng `result`, mỗi phần tử `[ID, topFiveAverage]`, **sắp theo ID tăng dần**.

**Ví dụ 1:**
```text
items = [[1,91],[1,92],[2,93],[2,97],[1,60],[2,77],[1,65],[1,87],[1,100],[2,100],[2,76]]
Output: [[1,87],[2,88]]
```
- Học sinh 1: điểm `91,92,60,65,87,100` → top 5 `100,92,91,87,65` → tổng `435` → `435 // 5 = 87`.
- Học sinh 2: điểm `93,97,77,100,76` → tổng `443` → `443 // 5 = 88` (88.6 bị cắt phần lẻ).

**Ví dụ 2:** mỗi học sinh 1 và 7 đều có năm điểm 100 → `[[1,100],[7,100]]`.

**Constraints (theo hiểu biết):**
- `1 <= items.length <= 1000`, `items[i].length == 2`
- `1 <= ID_i <= 1000`, `0 <= score_i <= 100`
- Mỗi ID có **ít nhất 5** điểm.

In [1]:
items = [[1, 91], [1, 92], [2, 93], [2, 97], [1, 60], [2, 77], [1, 65], [1, 87], [1, 100], [2, 100], [2, 76]]
s1 = [s for i, s in items if i == 1]
s2 = [s for i, s in items if i == 2]
print("HS 1:", s1, "-> top5:", sorted(s1, reverse=True)[:5], "-> tổng:", sum(sorted(s1, reverse=True)[:5]))
print("HS 2:", s2, "-> top5:", sorted(s2, reverse=True)[:5], "-> tổng:", sum(sorted(s2, reverse=True)[:5]))
print(435 // 5, 443 // 5, 443 / 5)

HS 1: [91, 92, 60, 65, 87, 100] -> top5: [100, 92, 91, 87, 65] -> tổng: 435
HS 2: [93, 97, 77, 100, 76] -> top5: [100, 97, 93, 77, 76] -> tổng: 443
87 88 88.6


## 2. Nền tảng Python cần biết

### 2.1. Gom nhóm bằng `dict`: key = ID, value = list điểm

Mẫu chuẩn khi key có thể **chưa tồn tại**:

```python
if sid not in groups:
    groups[sid] = []
groups[sid].append(score)
```

In [2]:
groups = {}
for sid, score in [[1, 91], [2, 93], [1, 92]]:
    if sid not in groups:
        groups[sid] = []
    groups[sid].append(score)
print(groups)       # {1: [91, 92], 2: [93]}

{1: [91, 92], 2: [93]}


### 2.2. `defaultdict(list)` — tự tạo list rỗng khi gặp key mới

`collections.defaultdict(list)` giống `dict`, nhưng khi truy cập key chưa có, nó **tự gán** `list()` (list rỗng). Bỏ được 2 dòng `if`.

In [3]:
from collections import defaultdict

groups = defaultdict(list)
for sid, score in [[1, 91], [2, 93], [1, 92]]:
    groups[sid].append(score)
print(dict(groups))
print(groups[99])          # key mới -> [] được tạo tự động
print(dict(groups))        # chú ý: 99 giờ đã nằm trong dict!

{1: [91, 92], 2: [93]}
[]
{1: [91, 92], 2: [93], 99: []}


### 2.3. Sắp giảm dần và lấy 5 phần tử đầu

- `sorted(a, reverse=True)` sắp **giảm dần**.
- `a[:5]` lấy tối đa 5 phần tử đầu (nếu ít hơn 5 thì lấy hết, không lỗi).

In [4]:
scores = [91, 92, 60, 65, 87, 100]
desc = sorted(scores, reverse=True)
print(desc, desc[:5], sum(desc[:5]))
print([1, 2][:5])          # không lỗi khi list ngắn

[100, 92, 91, 87, 65, 60] [100, 92, 91, 87, 65] 435
[1, 2]


### 2.4. Chia lấy nguyên `//` khác chia thường `/`

`443 / 5 = 88.6` (số thực), `443 // 5 = 88` (cắt phần lẻ, **làm tròn xuống**). Đề yêu cầu **integer division** → dùng `//`. Không dùng `round(88.6) = 89`!

In [5]:
print(443 / 5, 443 // 5, round(443 / 5), int(443 / 5))

88.6 88 89 88


### 2.5. Duyệt dict theo key tăng dần

`sorted(d)` trả list các **key** đã sắp. `sorted(d.items())` trả list cặp `(key, value)` đã sắp theo key.

In [6]:
d = {7: [100], 1: [90], 3: [80]}
print(sorted(d))
print(sorted(d.items()))

[1, 3, 7]
[(1, [90]), (3, [80]), (7, [100])]


## 3. Cách 1 — Gom nhóm + sắp từng nhóm (lời giải trực giác)

1. `groups = defaultdict(list)`; thêm mọi điểm vào đúng nhóm.
2. Với mỗi `sid` theo thứ tự tăng: `top5 = sorted(groups[sid], reverse=True)[:5]`.
3. Thêm `[sid, sum(top5) // 5]`.

In [7]:
def high_five_sort(items: list[list[int]]) -> list[list[int]]:
    groups = defaultdict(list)
    for sid, score in items:
        groups[sid].append(score)
    result = []
    for sid in sorted(groups):
        top5 = sorted(groups[sid], reverse=True)[:5]
        result.append([sid, sum(top5) // 5])
    return result

items1 = [[1, 91], [1, 92], [2, 93], [2, 97], [1, 60], [2, 77], [1, 65], [1, 87], [1, 100], [2, 100], [2, 76]]
items2 = [[1, 100], [7, 100], [1, 100], [7, 100], [1, 100], [7, 100], [1, 100], [7, 100], [1, 100], [7, 100]]
print(high_five_sort(items1))
print(high_five_sort(items2))

[[1, 87], [2, 88]]
[[1, 100], [7, 100]]


## 4. Trace cách 1 với ví dụ 1

| Bước | Trạng thái |
|---|---|
| Gom nhóm | `{1: [91,92,60,65,87,100], 2: [93,97,77,100,76]}` |
| HS 1 | sắp giảm `[100,92,91,87,65,60]` → top5 tổng `435` → `87` |
| HS 2 | sắp giảm `[100,97,93,77,76]` → tổng `443` → `88` |
| Kết quả | `[[1,87],[2,88]]` |

In [8]:
def trace_high_five(items: list[list[int]]) -> list[list[int]]:
    groups = defaultdict(list)
    for sid, score in items:
        groups[sid].append(score)
        print(f"đọc [{sid}, {score:>3}] -> nhóm {sid}: {groups[sid]}")
    result = []
    for sid in sorted(groups):
        desc = sorted(groups[sid], reverse=True)
        top5 = desc[:5]
        avg = sum(top5) // 5
        print(f"HS {sid}: giảm dần {desc} -> top5 {top5} -> {sum(top5)} // 5 = {avg}")
        result.append([sid, avg])
    print("Kết quả:", result)
    return result

trace_high_five(items1)

đọc [1,  91] -> nhóm 1: [91]
đọc [1,  92] -> nhóm 1: [91, 92]
đọc [2,  93] -> nhóm 2: [93]
đọc [2,  97] -> nhóm 2: [93, 97]
đọc [1,  60] -> nhóm 1: [91, 92, 60]
đọc [2,  77] -> nhóm 2: [93, 97, 77]
đọc [1,  65] -> nhóm 1: [91, 92, 60, 65]
đọc [1,  87] -> nhóm 1: [91, 92, 60, 65, 87]
đọc [1, 100] -> nhóm 1: [91, 92, 60, 65, 87, 100]
đọc [2, 100] -> nhóm 2: [93, 97, 77, 100]
đọc [2,  76] -> nhóm 2: [93, 97, 77, 100, 76]
HS 1: giảm dần [100, 92, 91, 87, 65, 60] -> top5 [100, 92, 91, 87, 65] -> 435 // 5 = 87
HS 2: giảm dần [100, 97, 93, 77, 76] -> top5 [100, 97, 93, 77, 76] -> 443 // 5 = 88
Kết quả: [[1, 87], [2, 88]]


[[1, 87], [2, 88]]

## 5. Nút thắt & insight: có cần sắp **toàn bộ** điểm của mỗi người?

Nếu một học sinh có `m` điểm, `sorted` tốn `O(m log m)`. Nhưng ta chỉ cần **5 điểm lớn nhất** — số 5 là **hằng số**.

**Insight:** giữ cho mỗi học sinh một "hộp" chỉ chứa **5 điểm lớn nhất đã thấy**. Gặp điểm mới:
- Nếu hộp chưa đủ 5 → bỏ vào.
- Nếu đủ 5 → so với **điểm nhỏ nhất trong hộp**; nếu điểm mới lớn hơn thì thay.

Cấu trúc phù hợp để "luôn lấy được phần tử nhỏ nhất nhanh" là **min-heap**.

### 5.1. Heap là gì? `heapq` trong Python

**Min-heap** là cấu trúc mà phần tử **nhỏ nhất luôn nằm ở `heap[0]`**. Python cài đặt heap trên `list` thường, qua module `heapq`:

| Hàm | Ý nghĩa | Chi phí |
|---|---|---|
| `heapq.heappush(h, x)` | thêm `x` | `O(log k)` |
| `heapq.heappop(h)` | lấy & xóa nhỏ nhất | `O(log k)` |
| `h[0]` | xem nhỏ nhất | `O(1)` |
| `heapq.heappushpop(h, x)` | đẩy `x` rồi lấy nhỏ nhất (gộp 1 bước) | `O(log k)` |

Với `k = 5` thì `log k` rất nhỏ → coi như `O(1)`.

In [9]:
import heapq

h = []
for x in [91, 60, 92, 65, 87]:
    heapq.heappush(h, x)
    print(f"push {x:>3} -> heap={h}, nhỏ nhất h[0]={h[0]}")
out = heapq.heappushpop(h, 100)        # đẩy 100, đá ra phần tử nhỏ nhất
print("heappushpop(100) đá ra:", out, "| heap còn:", sorted(h))

push  91 -> heap=[91], nhỏ nhất h[0]=91
push  60 -> heap=[60, 91], nhỏ nhất h[0]=60
push  92 -> heap=[60, 91, 92], nhỏ nhất h[0]=60
push  65 -> heap=[60, 65, 92, 91], nhỏ nhất h[0]=60
push  87 -> heap=[60, 65, 92, 91, 87], nhỏ nhất h[0]=60
heappushpop(100) đá ra: 60 | heap còn: [65, 87, 91, 92, 100]


Chú ý: in `h` ra **không** thấy dãy đã sắp — heap chỉ đảm bảo `h[0]` nhỏ nhất, phần còn lại sắp "một nửa". Muốn xem thứ tự thì `sorted(h)`.

## 6. Cách 2 (lời giải chính) — Min-heap cỡ 5 cho mỗi học sinh

Ghép từng đoạn:

**Đoạn 1:** `heaps = defaultdict(list)`.

**Đoạn 2:** với mỗi `[sid, score]`:
- `h = heaps[sid]`; nếu `len(h) < 5` → `heappush(h, score)`.
- Ngược lại nếu `score > h[0]` → `heapreplace(h, score)` (bỏ nhỏ nhất, thêm `score`).

**Đoạn 3:** kết quả `[[sid, sum(heaps[sid]) // 5] for sid in sorted(heaps)]`.

In [10]:
def high_five_heap(items: list[list[int]]) -> list[list[int]]:
    heaps = defaultdict(list)
    for sid, score in items:
        h = heaps[sid]
        if len(h) < 5:
            heapq.heappush(h, score)
        elif score > h[0]:
            heapq.heapreplace(h, score)     # pop nhỏ nhất rồi push score
    return [[sid, sum(heaps[sid]) // 5] for sid in sorted(heaps)]

print(high_five_heap(items1))
print(high_five_heap(items2))

[[1, 87], [2, 88]]
[[1, 100], [7, 100]]


### Trace heap của học sinh 1

Điểm theo thứ tự đọc: `91, 92, 60, 65, 87, 100`.

| Đọc | Heap trước (đã sắp để dễ nhìn) | Hành động | Heap sau |
|---:|---|---|---|
| 91 | `[]` | push | `[91]` |
| 92 | `[91]` | push | `[91,92]` |
| 60 | `[91,92]` | push | `[60,91,92]` |
| 65 | `[60,91,92]` | push | `[60,65,91,92]` |
| 87 | `[60,65,91,92]` | push (đủ 5) | `[60,65,87,91,92]` |
| 100 | `[60,...]` | `100 > 60` → thay 60 | `[65,87,91,92,100]` |

Tổng `435` → `87`.

In [11]:
def trace_heap_for(items: list[list[int]], target: int) -> None:
    h = []
    for sid, score in items:
        if sid != target:
            continue
        if len(h) < 5:
            heapq.heappush(h, score)
            act = "push"
        elif score > h[0]:
            old = heapq.heapreplace(h, score)
            act = f"thay {old}"
        else:
            act = f"bỏ qua (<= {h[0]})"
        print(f"đọc {score:>3}: {act:16} heap(sắp)={sorted(h)}")
    print(f"tổng = {sum(h)} -> {sum(h) // 5}")

trace_heap_for(items1, 1)

đọc  91: push             heap(sắp)=[91]
đọc  92: push             heap(sắp)=[91, 92]
đọc  60: push             heap(sắp)=[60, 91, 92]
đọc  65: push             heap(sắp)=[60, 65, 91, 92]
đọc  87: push             heap(sắp)=[60, 65, 87, 91, 92]
đọc 100: thay 60          heap(sắp)=[65, 87, 91, 92, 100]
tổng = 435 -> 87


## 7. Cách 3 — Sắp toàn bộ một lần theo `(ID tăng, điểm giảm)`

Mẹo **khóa sắp xếp tuple**: `key=lambda x: (x[0], -x[1])` sắp theo ID tăng; cùng ID thì `-score` tăng ⇔ `score` giảm.

Sau khi sắp, các điểm của mỗi học sinh nằm **liền nhau** và đã giảm dần. Duyệt một lượt, đếm `cnt` điểm đã lấy của ID hiện tại; chỉ cộng khi `cnt < 5`.

In [12]:
def high_five_global_sort(items: list[list[int]]) -> list[list[int]]:
    ordered = sorted(items, key=lambda x: (x[0], -x[1]))
    result = []
    cur_id, cnt, total = None, 0, 0
    for sid, score in ordered:
        if sid != cur_id:                     # sang học sinh mới
            if cur_id is not None:
                result.append([cur_id, total // 5])
            cur_id, cnt, total = sid, 0, 0
        if cnt < 5:
            total += score
            cnt += 1
    result.append([cur_id, total // 5])       # đừng quên học sinh cuối
    return result

print(sorted(items1, key=lambda x: (x[0], -x[1])))
print(high_five_global_sort(items1))
print(high_five_global_sort(items2))

[[1, 100], [1, 92], [1, 91], [1, 87], [1, 65], [1, 60], [2, 100], [2, 97], [2, 93], [2, 77], [2, 76]]
[[1, 87], [2, 88]]
[[1, 100], [7, 100]]


### Cách 3' — `itertools.groupby` sau khi sắp

`groupby(dãy_đã_sắp, key)` gom các phần tử **liền nhau** có cùng key. Kết hợp với dãy đã sắp theo `(id, -score)`.

In [13]:
from itertools import groupby

def high_five_groupby(items: list[list[int]]) -> list[list[int]]:
    ordered = sorted(items, key=lambda x: (x[0], -x[1]))
    result = []
    for sid, group in groupby(ordered, key=lambda x: x[0]):
        scores = [s for _, s in group]       # đã giảm dần
        result.append([sid, sum(scores[:5]) // 5])
    return result

print(high_five_groupby(items1))

[[1, 87], [2, 88]]


## 8. Cách 4 — `heapq.nlargest`

`heapq.nlargest(k, iterable)` trả `k` phần tử lớn nhất (giảm dần), bên trong dùng heap, `O(m log k)`. Rất gọn cho bản nộp.

In [14]:
print(heapq.nlargest(5, [91, 92, 60, 65, 87, 100]))

def high_five_nlargest(items: list[list[int]]) -> list[list[int]]:
    groups = defaultdict(list)
    for sid, score in items:
        groups[sid].append(score)
    return [[sid, sum(heapq.nlargest(5, groups[sid])) // 5] for sid in sorted(groups)]

print(high_five_nlargest(items1))

[100, 92, 91, 87, 65]
[[1, 87], [2, 88]]


## 9. Code nộp LeetCode

### Bản heap (lời giải chính)

In [15]:
import heapq
from collections import defaultdict

class Solution:
    def highFive(self, items: list[list[int]]) -> list[list[int]]:
        heaps = defaultdict(list)
        for sid, score in items:
            h = heaps[sid]
            if len(h) < 5:
                heapq.heappush(h, score)
            elif score > h[0]:
                heapq.heapreplace(h, score)
        return [[sid, sum(heaps[sid]) // 5] for sid in sorted(heaps)]

sol = Solution()
assert sol.highFive(items1) == [[1, 87], [2, 88]]
assert sol.highFive(items2) == [[1, 100], [7, 100]]
print("Solution (heap) PASS")

Solution (heap) PASS


### Bản sắp từng nhóm (dễ nhớ)

In [16]:
class SolutionSort:
    def highFive(self, items: list[list[int]]) -> list[list[int]]:
        groups = defaultdict(list)
        for sid, score in items:
            groups[sid].append(score)
        return [[sid, sum(sorted(groups[sid], reverse=True)[:5]) // 5] for sid in sorted(groups)]

assert SolutionSort().highFive(items1) == [[1, 87], [2, 88]]
assert SolutionSort().highFive(items2) == [[1, 100], [7, 100]]
print("SolutionSort PASS")

SolutionSort PASS


## 10. Vì sao đúng?

**Cách 1:** Sau khi gom, `groups[sid]` chứa **đúng mọi** điểm của học sinh `sid` (mỗi item được thêm đúng một lần vào đúng nhóm). Sắp giảm và lấy 5 đầu = 5 điểm lớn nhất. Chia `// 5` đúng định nghĩa. Duyệt `sorted(groups)` cho thứ tự ID tăng.

**Cách 2 — invariant:** sau khi xử lý một số item, `heaps[sid]` chứa **5 điểm lớn nhất** (hoặc tất cả nếu chưa đủ 5) trong các điểm của `sid` **đã đọc**.
- Chưa đủ 5: thêm vào — giữ invariant.
- Đủ 5, điểm mới `x`:
  - `x <= h[0]` (nhỏ nhất trong top 5): `x` không thể vào top 5 → bỏ qua đúng.
  - `x > h[0]`: top 5 mới = top 5 cũ bỏ `h[0]` thêm `x` → đúng việc `heapreplace` làm.

Đọc hết ⇒ mỗi heap là top 5 thực sự. (Đề đảm bảo mỗi ID có ≥ 5 điểm nên heap luôn đủ 5.)

## 11. Độ phức tạp

Gọi `N = len(items)`, `S` = số học sinh khác nhau, `k = 5`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Gom + sắp từng nhóm | `O(N log N)` worst (một nhóm lớn) + `O(S log S)` | `O(N)` |
| 2. Min-heap cỡ 5 | `O(N log k)` = `O(N)` + `O(S log S)` | `O(S · k)` = `O(S)` |
| 3. Sắp toàn bộ theo `(id, -score)` | `O(N log N)` | `O(N)` |
| 4. `nlargest` | `O(N log k)` + `O(S log S)` | `O(N)` (vẫn gom hết điểm) |

Cách 2 tối ưu cả thời gian lẫn bộ nhớ: mỗi học sinh chỉ giữ 5 số.

## 12. Bộ test

Nhóm test:
1. Hai ví dụ đề.
2. Một học sinh đúng 5 điểm.
3. Nhiều điểm trùng nhau; điểm 0.
4. ID không liên tục, đưa vào lộn xộn (`[3, ...]` trước `[1, ...]`).
5. Trung bình không nguyên → kiểm tra **làm tròn xuống**.
6. Đối chiếu ngẫu nhiên 6 cách.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:24} PASS -> {actual}")

tests = [
    ("đề 1", items1, [[1, 87], [2, 88]]),
    ("đề 2", items2, [[1, 100], [7, 100]]),
    ("1 HS đúng 5 điểm", [[5, 10], [5, 20], [5, 30], [5, 40], [5, 50]], [[5, 30]]),
    ("điểm 0 và trùng", [[1, 0]] * 4 + [[1, 7], [1, 7]], [[1, 2]]),
    ("ID lộn xộn", [[3, 50]] * 5 + [[1, 60]] * 5, [[1, 60], [3, 50]]),
    ("làm tròn xuống", [[9, 100], [9, 100], [9, 100], [9, 100], [9, 99]], [[9, 99]]),
]
for name, it, exp in tests:
    check(name, high_five_heap(it), exp)

đề 1                     PASS -> [[1, 87], [2, 88]]
đề 2                     PASS -> [[1, 100], [7, 100]]
1 HS đúng 5 điểm         PASS -> [[5, 30]]
điểm 0 và trùng          PASS -> [[1, 2]]
ID lộn xộn               PASS -> [[1, 60], [3, 50]]
làm tròn xuống           PASS -> [[9, 99]]


In [18]:
import random
random.seed(1086)
for _ in range(1000):
    ids = random.sample(range(1, 30), random.randint(1, 6))
    it = []
    for sid in ids:
        for _ in range(random.randint(5, 12)):
            it.append([sid, random.randint(0, 100)])
    random.shuffle(it)
    r = [high_five_sort(it), high_five_heap(it), high_five_global_sort(it),
         high_five_groupby(it), high_five_nlargest(it), SolutionSort().highFive(it)]
    assert all(x == r[0] for x in r), (it, r)
print("1000 test ngẫu nhiên: 6 cách khớp nhau")

1000 test ngẫu nhiên: 6 cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `/` hoặc `round` thay vì `//`

In [19]:
def wrong_round(items):
    groups = defaultdict(list)
    for sid, score in items:
        groups[sid].append(score)
    return [[sid, round(sum(sorted(groups[sid], reverse=True)[:5]) / 5)] for sid in sorted(groups)]

print("sai :", wrong_round(items1), "| đúng:", high_five_sort(items1))

sai : [[1, 87], [2, 89]] | đúng: [[1, 87], [2, 88]]


### Lỗi 2: Sắp tăng dần rồi lấy `[:5]` → lấy 5 điểm **thấp nhất**

In [20]:
def wrong_ascending(items):
    groups = defaultdict(list)
    for sid, score in items:
        groups[sid].append(score)
    return [[sid, sum(sorted(groups[sid])[:5]) // 5] for sid in sorted(groups)]

print("sai :", wrong_ascending(items1), "| đúng:", high_five_sort(items1))

sai : [[1, 79], [2, 88]] | đúng: [[1, 87], [2, 88]]


**Sửa:** `sorted(..., reverse=True)[:5]` hoặc `sorted(...)[-5:]`.

### Lỗi 3: Dùng **max-heap** sai cách / so `score > h[-1]`

`h[-1]` **không** phải phần tử lớn hay nhỏ nhất trong heap. Chỉ `h[0]` có ý nghĩa (nhỏ nhất với min-heap).

### Lỗi 4: Quên sắp kết quả theo ID

`dict` giữ thứ tự **chèn**, không phải thứ tự key. Nếu input có ID 3 trước ID 1 thì kết quả sai thứ tự.

In [21]:
def wrong_no_sort_ids(items):
    groups = defaultdict(list)
    for sid, score in items:
        groups[sid].append(score)
    return [[sid, sum(sorted(g, reverse=True)[:5]) // 5] for sid, g in groups.items()]

it = [[3, 50]] * 5 + [[1, 60]] * 5
print("sai :", wrong_no_sort_ids(it), "| đúng:", high_five_sort(it))

sai : [[3, 50], [1, 60]] | đúng: [[1, 60], [3, 50]]


### Lỗi 5: Cách 3 quên thêm học sinh **cuối cùng** sau vòng lặp

Mẫu "phát hiện đổi nhóm" chỉ ghi kết quả khi sang nhóm mới — nhóm cuối không có "nhóm mới" theo sau, nên phải `append` thêm một lần sau vòng `for`.

## 14. Mẹo nhận dạng pattern

- "Theo **từng** người/ID/nhóm" → **gom nhóm** bằng `dict`/`defaultdict(list)`.
- "**k** phần tử lớn nhất/nhỏ nhất" với `k` nhỏ → **heap cỡ k** (top-k): giữ **min-heap** để lấy **k lớn nhất**, và ngược lại.
- "Kết quả theo key tăng dần" → `sorted(groups)` ở bước cuối.
- Sắp nhiều tiêu chí → `key=lambda x: (tiêu_chí_1, -tiêu_chí_2)`.

| Bài liên quan | Ý tưởng |
|---|---|
| 215. Kth Largest Element in an Array | min-heap cỡ k |
| 347. Top K Frequent Elements | đếm + heap |
| 703. Kth Largest Element in a Stream | heap cỡ k cập nhật dần |
| 1636. Sort Array by Increasing Frequency | sắp theo tuple key |

## 15. Biến thể

### Biến thể A — Top-k tổng quát và trung bình số thực

In [22]:
def top_k_average(items: list[list[int]], k: int) -> dict:
    heaps = defaultdict(list)
    for sid, score in items:
        h = heaps[sid]
        if len(h) < k:
            heapq.heappush(h, score)
        elif score > h[0]:
            heapq.heapreplace(h, score)
    return {sid: round(sum(h) / len(h), 2) for sid, h in sorted(heaps.items())}

print(top_k_average(items1, 3))
print(top_k_average(items1, 5))

{1: 94.33, 2: 96.67}
{1: 87.0, 2: 88.6}


### Biến thể B — Xếp hạng học sinh theo điểm top-5 (cao trước, hòa thì ID nhỏ trước)

In [23]:
def rank_students(items: list[list[int]]) -> list[int]:
    res = high_five_heap(items)
    return [sid for sid, avg in sorted(res, key=lambda x: (-x[1], x[0]))]

print(rank_students(items1))
print(rank_students([[3, 50]] * 5 + [[1, 50]] * 5 + [[2, 90]] * 5))

[2, 1]
[2, 1, 3]


## 16. Bài tập tự luyện (tăng dần)

**Bài 1 (tính tay):** HS 4 có điểm `70, 80, 90, 85, 75, 95, 60`. Top 5 là gì, trung bình (chia nguyên) bao nhiêu?

**Bài 2 (điền chỗ trống):** hoàn thiện `ex_group_scores(items)` trả về `dict` ID → list điểm (dùng `dict` thường, **không** dùng `defaultdict`).

**Bài 3 (viết lại):** `ex_high_five(items)` dùng `ex_group_scores` + sắp giảm.

**Bài 4 (heap):** `ex_kth_largest(nums, k)` — phần tử lớn thứ `k` bằng min-heap cỡ `k` (LeetCode 215).

**Bài 5 (mini challenge):** `ex_low_five(items)` — trung bình (chia nguyên) của **5 điểm thấp nhất** mỗi học sinh, dùng **heap** (gợi ý: max-heap mô phỏng bằng cách lưu `-score`).

In [24]:
def ex_group_scores(items: list[list[int]]) -> dict:
    groups = {}
    for sid, score in items:
        # TODO 1: nếu sid chưa có trong groups -> tạo list rỗng
        # TODO 2: thêm score
        pass
    return groups

# assert ex_group_scores([[1, 5], [2, 6], [1, 7]]) == {1: [5, 7], 2: [6]}

In [25]:
def ex_high_five(items: list[list[int]]) -> list[list[int]]:
    # TODO: gom nhóm; với sid tăng dần: sum(top5) // 5
    pass

# assert ex_high_five(items1) == [[1, 87], [2, 88]]

In [26]:
def ex_kth_largest(nums: list[int], k: int) -> int:
    # TODO: heap cỡ k; trả h[0]
    pass

# assert ex_kth_largest([3, 2, 1, 5, 6, 4], 2) == 5
# assert ex_kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == 4

In [27]:
def ex_low_five(items: list[list[int]]) -> list[list[int]]:
    # TODO: mỗi sid một heap chứa -score cỡ 5; khi đủ 5, nếu score < -h[0] thì thay
    pass

# assert ex_low_five(items1) == [[1, 79], [2, 88]]

## 17. Gợi ý 3 mức

**Bài 1:** (1) Sắp giảm. (2) `95, 90, 85, 80, 75, 70, 60` → top 5 `95,90,85,80,75`. (3) Tổng `425` → `85`.

**Bài 2:** (1) Mẫu gom nhóm mục 2.1. (2) `if sid not in groups: groups[sid] = []`. (3) `groups[sid].append(score)`.

**Bài 3:** (1) Như cách 1. (2) `sorted(groups[sid], reverse=True)[:5]`. (3) Duyệt `sorted(groups)`.

**Bài 4:** (1) Giữ k số lớn nhất. (2) `heappush` khi chưa đủ k; `heapreplace` khi `x > h[0]`. (3) Phần tử nhỏ nhất của top-k chính là lớn thứ k: `h[0]`.

**Bài 5:** (1) Muốn giữ 5 **nhỏ nhất** → cần biết phần tử **lớn nhất** trong hộp → max-heap. (2) `heapq` chỉ có min-heap, nên lưu `-score`: lớn nhất trong hộp là `-h[0]`. (3) Kết quả tổng `-sum(h)`, chia `// 5`.

## 18. Đáp án tham khảo + template + checklist

Tự làm trước rồi mới xem. Bài 1: top 5 `[95, 90, 85, 80, 75]`, trung bình `85`. (Bài 5 với ví dụ 1: HS 1 năm điểm thấp nhất `60,65,87,91,92` → `395 // 5 = 79`.)

In [28]:
def ans_group_scores(items):
    groups = {}
    for sid, score in items:
        if sid not in groups:
            groups[sid] = []
        groups[sid].append(score)
    return groups

def ans_high_five(items):
    groups = ans_group_scores(items)
    return [[sid, sum(sorted(groups[sid], reverse=True)[:5]) // 5] for sid in sorted(groups)]

def ans_kth_largest(nums, k):
    h = []
    for x in nums:
        if len(h) < k:
            heapq.heappush(h, x)
        elif x > h[0]:
            heapq.heapreplace(h, x)
    return h[0]

def ans_low_five(items):
    heaps = defaultdict(list)
    for sid, score in items:
        h = heaps[sid]
        if len(h) < 5:
            heapq.heappush(h, -score)
        elif score < -h[0]:
            heapq.heapreplace(h, -score)
    return [[sid, -sum(heaps[sid]) // 5] for sid in sorted(heaps)]

assert sum(sorted([70, 80, 90, 85, 75, 95, 60], reverse=True)[:5]) // 5 == 85
assert ans_group_scores([[1, 5], [2, 6], [1, 7]]) == {1: [5, 7], 2: [6]}
assert ans_high_five(items1) == [[1, 87], [2, 88]]
assert ans_kth_largest([3, 2, 1, 5, 6, 4], 2) == 5
assert ans_kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == 4
assert ans_low_five(items1) == [[1, 79], [2, 88]]
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
import heapq
from collections import defaultdict

class Solution:
    def highFive(self, items: list[list[int]]) -> list[list[int]]:
        heaps = defaultdict(list)                  # ID -> min-heap 5 điểm lớn nhất
        for sid, score in items:
            h = heaps[sid]
            if len(h) < 5:
                heapq.heappush(h, score)
            elif score > h[0]:                     # lớn hơn điểm yếu nhất trong top 5
                heapq.heapreplace(h, score)
        return [[sid, sum(heaps[sid]) // 5] for sid in sorted(heaps)]
```

### Checklist trước khi submit
- [ ] Gom đúng điểm theo ID.
- [ ] Lấy 5 điểm **cao nhất** (sắp giảm, hoặc min-heap cỡ 5).
- [ ] Chia `// 5` (làm tròn xuống), không `/`, không `round`.
- [ ] Kết quả sắp theo **ID tăng dần**.
- [ ] Trả `list[list[int]]` dạng `[[ID, avg], ...]`.
- [ ] Heap: Time `O(N + S log S)`, Space `O(S)`.